1. Problem Statement and Study Goal

- Problem Statement: Standard IT metrics like Mean Time to Resolution (MTTR) introduce severe statistical bias because they cannot account for open tickets, which artificially skews the operational picture of help desk efficiency.
- Study Goal: This study applies non-parametric survival analysis via Kaplan-Meier curves to accurately measure and benchmark the time-to-resolution operational effectiveness of high-volume, established support groups.
- Dataset Description: The analysis utilizes the IT Service Management (ITSM) dataset from the UCI Machine Learning Repository, which tracks historical incident lifecycles under the ITIL framework.
- Dataset Source: The data is sourced from a large IT service provider's ServiceNow logging platform, capturing multi-row state updates, assignments, and timestamps for tracking ticket lifecycles.

2. Rationale and Criteria for Selecting Support Groups

- Target Variable Choice: The analysis uses the last registered support group and its final state timestamp for each unique ticket.
- Operational Rationale: Attributing the ticket to the final assignment group directly evaluates that group's capacity to successfully resolve and exit the incident from the active queue.
- Analytical Choice: This choice measures the total elapsed duration from a ticket's opening time until closure by that final group, answering how long tickets ultimately landing in that domain take to resolve.
- Placeholder Removal: All unassigned or incomplete tracking categories labeled as "?" are discarded from the study because they do not represent active human support teams.
- Volume Threshold Filter: A strict inclusion rule is applied requiring a group to have at least 20 closed tickets within the observation period.
- Volume Threshold Rationale: This minimum closure baseline controls statistical variance and ensures the Kaplan-Meier curve has sufficient events to calculate stable survival probabilities.
- Censoring Guard Filter: A maximum censoring boundary is enforced where the proportion of open tickets to total tickets must be less than or equal to 0.70.
- Censoring Guard Rationale: This rule guarantees that the survival curve drops far enough below the 0.50 Y-axis mark, enabling the estimator to calculate a valid, unambiguous median survival time for benchmarking.

3. Attribute Filters and Data Processing Algorithm Sketch

- Temporal Window Filter: The observation boundaries are defined by finding the absolute maximum timestamp in the data (`T_end`) and calculating exactly one year backward (`T_start = T_end - 365 days`).
- Subject Entry Filter: Only tickets with an `opened_at` timestamp strictly falling between `T_start` and `T_end` are included to prevent left-truncation and immortal time bias.

Step-by-step sketch:
- Step 1 (Boundary Setting): Identify the absolute maximum dataset timestamp (`T_end`) and subtract 365 days to establish the strict rolling 1-year study window (`T_start`).
- Step 2 (Data Cleaning): Drop all rows containing unassigned `"?"` values within the `assignment_group` attribute.
- Step 3 (Window Isolation): Filter the rows to isolate only those incidents whose original opening timestamp falls within the `[T_start, T_end]` window.
- Step 4 (Event Mapping): Map the event indicator `E` per ticket based on its final operational state at `T_end`, assigning `E = 1` for Closed/Resolved states and `E = 0` (censored) for all active states (e.g., New, Active, On Hold).
- Step 5 (Duration Calculation): Dynamically compute the duration `T` for the subject row. If `E = 1`, `T = Closed Time − Opening Time`. If `E = 0`, `T = T_end − Opening Time`.


In [1]:
import inspect
import sys
from pathlib import Path

cwd = Path.cwd()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
sys.path.insert(0, str(repo_root))

import pandas as pd
from featurization import get_package_info, notebook_utils as feat_notebook_utils
from featurization_scripts.featurization import ticket_survival_summary

notebook_dir = repo_root / "notebooks"
if not notebook_dir.exists():
    raise FileNotFoundError("Could not locate notebooks directory to build resolver")

resolver = feat_notebook_utils.build_notebook_resolver(str(notebook_dir))

raw_clean_dataset = feat_notebook_utils.load_featurization_input_dataset(resolver)
if raw_clean_dataset is None:
    if getattr(resolver, "featurization_input_path", None):
        raw_clean_dataset = pd.read_csv(resolver.featurization_input_path)
    else:
        raise FileNotFoundError(
            "Could not resolve the input dataset path from the featurization resolver"
        )

survival_df = raw_clean_dataset.copy()

for col in ["opened_at", "closed_at"]:
    if col not in survival_df.columns:
        raise KeyError(f"{col} column is required for survival preprocessing")
    survival_df[col] = pd.to_datetime(survival_df[col], errors="coerce")

ticket_col = (
    "ticket_number"
    if "ticket_number" in survival_df.columns
    else ("number" if "number" in survival_df.columns else None)
)
if ticket_col is None:
    raise KeyError("ticket number column is required (ticket_number or number)")

if "assignment_group" not in survival_df.columns:
    raise KeyError("assignment_group column is required in the clean dataset")

survival_df["assignment_group"] = survival_df["assignment_group"].fillna("?").astype(str)
survival_df = survival_df.loc[survival_df["assignment_group"] != "?"].copy()

datetime_cols = [col for col in survival_df.columns if survival_df[col].dtype.kind == "M"]
if not datetime_cols:
    raise ValueError("No datetime columns found for observation boundary calculation")
observation_end = survival_df[datetime_cols].max().max()
observation_start = observation_end - pd.DateOffset(days=365)

window_df = survival_df.loc[
    (survival_df["opened_at"] >= observation_start)
    & (survival_df["opened_at"] <= observation_end)
].copy()

state_col = next((c for c in ["incident_state", "state", "status"] if c in window_df.columns), None)
closed_states = {"closed", "resolved"}
if state_col is not None:
    window_df["survival_event"] = (
        window_df[state_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(closed_states)
        .astype(int)
    )
else:
    window_df["survival_event"] = window_df["closed_at"].notna().astype(int)

window_df["survival_time"] = (
    window_df["closed_at"].fillna(observation_end) - window_df["opened_at"]
)
window_df["survival_time_days"] = window_df["survival_time"].dt.total_seconds() / 86400.0

closed_candidates = window_df.loc[window_df["survival_event"] == 1].copy()
group_closed_counts = (
    closed_candidates.groupby("assignment_group")[ticket_col]
    .nunique()
    .reset_index(name="closed_ticket_count")
)
eligible_groups = set(
    group_closed_counts.loc[group_closed_counts["closed_ticket_count"] >= 20, "assignment_group"]
)

group_stats = (
    window_df.groupby("assignment_group")
    .agg(
        closed_tickets=("survival_event", lambda x: int((x == 1).sum())),
        open_tickets=("survival_event", lambda x: int((x == 0).sum())),
    )
    .reset_index()
)
group_stats["total_tickets"] = group_stats["open_tickets"] + group_stats["closed_tickets"]
group_stats["open_ratio"] = group_stats["open_tickets"] / group_stats["total_tickets"]

elegible_groups = eligible_groups & set(
    group_stats.loc[group_stats["open_ratio"] <= 0.70, "assignment_group"]
)

final_survival_df = window_df.loc[window_df["assignment_group"].isin(eligible_groups)].copy()
final_survival_df = final_survival_df.reset_index(drop=True)


def _call_with_supported_kwargs(func, *args, **kwargs):
    sig = inspect.signature(func)
    supported_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
    if not supported_kwargs and args and len(sig.parameters) == 1:
        return func(args[0])
    return func(**supported_kwargs)


try:
    survival_subject_df = _call_with_supported_kwargs(
        ticket_survival_summary,
        final_survival_df,
        ticket_col=ticket_col,
        opened_at_col="opened_at",
        closed_at_col="closed_at",
        state_col=state_col,
        assignment_group_col="assignment_group",
        event_col="survival_event",
        time_col="survival_time_days",
        observation_end=observation_end,
        closed_states=closed_states,
    )
except TypeError:
    survival_subject_df = (
        final_survival_df.groupby(ticket_col, as_index=False)
        .agg(
            assignment_group=("assignment_group", "first"),
            opened_at=("opened_at", "min"),
            closed_at=("closed_at", "max"),
            survival_event=("survival_event", "max"),
            survival_time_days=("survival_time_days", "max"),
        )
    )
    survival_subject_df["survival_time"] = (
        survival_subject_df["closed_at"].fillna(observation_end) - survival_subject_df["opened_at"]
    )
    survival_subject_df["survival_time_days"] = (
        survival_subject_df["survival_time"].dt.total_seconds() / 86400.0
    )

print("Observation start:", observation_start)
print("Observation end:", observation_end)
print("Candidate rows:", len(window_df))
print("Retained groups:", len(eligible_groups))
print("Modeling-ready tickets:", len(survival_subject_df))

output_path = getattr(resolver, "model_ready_dataset_path", None)
if output_path is None:
    raise KeyError("Could not resolve model-ready output path from the featurization resolver")

output_path = Path(output_path)
output_path.parent.mkdir(parents=True, exist_ok=True)
survival_subject_df.to_csv(output_path, index=False)
print("Saved modeling-ready survival dataset to", output_path)


Observation start: 2016-02-19 15:00:00
Observation end: 2017-02-18 15:00:00
Candidate rows: 127499
Retained groups: 52
Modeling-ready tickets: 24509
Saved modeling-ready survival dataset to /home/rajiv/programming/kmds_migration/itsm_analysis/data/featurization/itsm_survival_model_ready_numeric_data.csv


In [2]:
final_survival_df

,number,incident_state,opened_at,assignment_group,resolved_at,closed_at,survival_event,survival_time,survival_time_days
0,INC0000045,New,2016-02-29 01:16:00,Group 56,2016-02-29 11:29:00,2016-03-05 12:00:00,0,5 days 10:44:00,5.447222
1,INC0000045,Resolved,2016-02-29 01:16:00,Group 56,2016-02-29 11:29:00,2016-03-05 12:00:00,1,5 days 10:44:00,5.447222
2,INC0000045,Resolved,2016-02-29 01:16:00,Group 56,2016-02-29 11:29:00,2016-03-05 12:00:00,1,5 days 10:44:00,5.447222
3,INC0000045,Closed,2016-02-29 01:16:00,Group 56,2016-02-29 11:29:00,2016-03-05 12:00:00,1,5 days 10:44:00,5.447222
4,INC0000047,New,2016-02-29 04:40:00,Group 70,2016-03-01 09:52:00,2016-03-06 10:00:00,0,6 days 05:20:00,6.222222
...,...,...,...,...,...,...,...,...,...
126392,INC0120835,Closed,2017-02-16 09:09:00,Group 31,2017-02-16 09:53:00,2017-02-16 09:53:00,1,0 days 00:44:00,0.030556
126393,INC0121064,Active,2017-02-16 14:17:00,Group 70,2017-02-16 16:38:00,2017-02-16 16:38:00,0,0 days 02:21:00,0.097917
126394,INC0121064,Active,2017-02-16 14:17:00,Group 31,2017-02-16 16:38:00,2017-02-16 16:38:00,0,0 days 02:21:00,0.097917
126395,INC0121064,Resolved,2017-02-16 14:17:00,Group 31,2017-02-16 16:38:00,2017-02-16 16:38:00,1,0 days 02:21:00,0.097917


In [3]:
open_count = int((final_survival_df["survival_event"] == 0).sum())
closed_count = int((final_survival_df["survival_event"] == 1).sum())

print(f"Open tickets in survival dataset: {open_count}")
print(f"Closed tickets in survival dataset: {closed_count}")

Open tickets in survival dataset: 80332
Closed tickets in survival dataset: 46065
